In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
# 自动定位项目根目录：notebook 位于 notes/ 下，向上一级即为项目根
# 如果 Jupyter 已经从项目根目录启动，则无需跳转
# 后续 'data/ShanghaiPM_Training.csv' 等相对路径即可在任何环境中正确解析
cwd = os.getcwd()
if os.path.basename(cwd) == 'notes':
    os.chdir('..')
print('当前工作目录:', os.getcwd())

当前工作目录: d:\github\prediction


In [2]:
df_raw = pd.read_csv('data/ShanghaiPM_Training.csv', na_values='NA')
df_raw['datetime'] = pd.to_datetime(
    {'year': df_raw['year'], 'month': df_raw['month'], 'day': df_raw['day'], 'hour': df_raw['hour']}
)
df_raw = df_raw.set_index('datetime')
df_raw = df_raw.sort_index()

pm_cols = ['PM_Jingan', 'PM_US Post', 'PM_Xuhui']
df_raw['pm_ave'] = df_raw[pm_cols].mean(axis=1)
df_raw.head()

,No,year,month,day,hour,season,PM_Jingan,PM_US Post,PM_Xuhui,DEWP,HUMI,PRES,TEMP,cbwd,Iws,precipitation,Iprec,pm_ave
datetime,,,,,,,,,,,,,,,,,,
2010-01-01 00:00:00,1,2010,1,1,0,4,NaN,NaN,NaN,-6.0,59.48,1026.1,1.0,cv,1.0,0.0,0.0,NaN
2010-01-01 01:00:00,2,2010,1,1,1,4,NaN,NaN,NaN,-6.0,59.48,1025.1,1.0,SE,2.0,0.0,0.0,NaN
2010-01-01 02:00:00,3,2010,1,1,2,4,NaN,NaN,NaN,-7.0,59.21,1025.1,0.0,SE,4.0,0.0,0.0,NaN
2010-01-01 03:00:00,4,2010,1,1,3,4,NaN,NaN,NaN,-6.0,63.94,1024.0,0.0,SE,5.0,0.0,0.0,NaN
2010-01-01 04:00:00,5,2010,1,1,4,4,NaN,NaN,NaN,-6.0,63.94,1023.0,0.0,SE,8.0,0.0,0.0,NaN


In [8]:
KEEP_COLS = [
    'pm_ave',       # 徐汇站 PM2.5 浓度 (ug/m3)
    'TEMP',           # 气温 (摄氏度)
    'HUMI',           # 相对湿度 (%)
]

df_train= df_raw[KEEP_COLS].copy()
df_train= df_train.dropna(subset=['pm_ave'])

df_train.head(10)




,pm_ave,TEMP,HUMI
datetime,,,
2011-12-28 18:00:00,36.0,11.0,62.00
2011-12-28 19:00:00,41.0,11.0,62.00
2011-12-28 20:00:00,44.0,10.0,71.07
2011-12-28 21:00:00,40.0,10.0,71.07
2011-12-28 22:00:00,25.0,10.0,76.18
2011-12-28 23:00:00,28.0,10.0,76.18
2011-12-29 00:00:00,34.0,10.0,76.18
2011-12-29 01:00:00,25.0,9.0,76.01
2011-12-29 02:00:00,27.0,9.0,76.01


In [9]:
print('\n统计摘要:')
df_train.describe().round(2)


统计摘要:


,pm_ave,TEMP,HUMI
count,34394.00,34389.00,34389.00
mean,54.40,17.63,69.93
std,44.12,9.17,17.77
min,1.00,-4.00,13.09
25%,25.67,10.00,58.29
50%,42.00,19.00,72.96
75%,69.00,25.00,83.60
max,730.00,41.00,100.00


In [ ]:
# 读取测试集，构造时间戳 (na_values='NA' 把字符串 NA 转成 NaN)
df_test = pd.read_csv('data/ShanghaiPM_Test.csv', na_values='NA')#注意现在我们有df_prediction,df_train,df_raw三个数据集，df_prediction是测试集，df_train是训练集，df_raw是原始数据集
df_test['timestamp'] = pd.to_datetime(df_test[['year', 'month', 'day', 'hour']])
df_test = df_test.dropna(subset=['pm_ave'])
df_test.head(10)

,No,year,month,day,hour,season,PM_Jingan,PM_US Post,PM_Xuhui,DEWP,HUMI,PRES,TEMP,cbwd,Iws,precipitation,Iprec,timestamp
0,52184,2015,12,15,7,4,269.0,269.0,276.0,1,70.27,1025,6,NW,176,0.0,0.0,2015-12-15 07:00:00
1,52185,2015,12,15,8,4,261.0,251.0,271.0,0,61.01,1025,7,NW,181,0.0,0.0,2015-12-15 08:00:00
2,52186,2015,12,15,9,4,247.0,238.0,266.0,1,61.26,1026,8,NW,184,0.0,0.0,2015-12-15 09:00:00
3,52187,2015,12,15,10,4,236.0,224.0,243.0,1,61.26,1026,8,NW,188,0.0,0.0,2015-12-15 10:00:00
4,52188,2015,12,15,11,4,202.0,188.0,214.0,-1,49.51,1026,9,NW,194,0.0,0.0,2015-12-15 11:00:00
5,52189,2015,12,15,12,4,189.0,195.0,194.0,-1,46.29,1025,10,NW,200,0.0,0.0,2015-12-15 12:00:00
6,52190,2015,12,15,13,4,198.0,193.0,203.0,-1,46.29,1024,10,NW,206,0.0,0.0,2015-12-15 13:00:00
7,52191,2015,12,15,14,4,202.0,186.0,198.0,0,49.80,1024,10,NW,212,0.0,0.0,2015-12-15 14:00:00
8,52192,2015,12,15,15,4,176.0,157.0,181.0,-1,46.29,1024,10,NW,216,0.0,0.0,2015-12-15 15:00:00
9,52193,2015,12,15,16,4,180.0,176.0,165.0,-1,46.29,1024,10,NW,221,0.0,0.0,2015-12-15 16:00:00


In [18]:
df_prediction = pd.DataFrame({
    'timestamp': df_test['timestamp'],
    'predicted_pm_ave': df_test['timestamp'].dt.hour.map(hourly_baseline)
})

print(f'\n预测表行数: {len(df_prediction)}')
df_prediction.head(10)


预测表行数: 401


,timestamp,predicted_pm_ave
0,2015-12-15 07:00:00,55.532867
1,2015-12-15 08:00:00,56.999767
2,2015-12-15 09:00:00,56.732380
3,2015-12-15 10:00:00,55.710901
4,2015-12-15 11:00:00,54.634949
5,2015-12-15 12:00:00,54.408572
6,2015-12-15 13:00:00,53.767434
7,2015-12-15 14:00:00,53.070396
8,2015-12-15 15:00:00,53.451964
9,2015-12-15 16:00:00,53.110179


In [ ]:
# 从测试集算出每个时间点的真实 pm_ave (三站均值，至少一个有效)
pm_cols = ['PM_Jingan', 'PM_US Post', 'PM_Xuhui']
df_test['pm_ave'] = df_test[pm_cols].mean(axis=1)

# 把真实值并进 df_prediction (按行对齐，两者时间戳顺序一致)
df_prediction['actual_pm_ave'] = df_test['pm_ave'].values

# 逐条比对: 残差 = 真实值 - 预测值 (每个测试点单独算，不先做小时平均)
df_prediction['error'] = df_prediction['actual_pm_ave'] - df_prediction['predicted_pm_ave']

# 只保留有真实值的点参与 loss (三站全缺失的测试点无法评估)
mask = df_prediction['actual_pm_ave'].notna()
err = df_prediction.loc[mask, 'error']#只跳出mask=true的那几行

# 均方误差 MSE = 残差平方的均值
mse = (err ** 2).mean()

print(f'有效比对样本数: {mask.sum()} / {len(df_prediction)}')
print(f'MSE = {mse:.2f}')
df_prediction.head(300)


有效比对样本数: 401 / 401
MSE = 5477.94


,timestamp,predicted_pm_ave,actual_pm_ave,error
0,2015-12-15 07:00:00,55.532867,271.333333,215.800466
1,2015-12-15 08:00:00,56.999767,261.000000,204.000233
2,2015-12-15 09:00:00,56.732380,250.333333,193.600954
3,2015-12-15 10:00:00,55.710901,234.333333,178.622432
4,2015-12-15 11:00:00,54.634949,201.333333,146.698385
...,...,...,...,...
295,2015-12-27 14:00:00,53.070396,41.000000,-12.070396
296,2015-12-27 15:00:00,53.451964,38.333333,-15.118630
297,2015-12-27 16:00:00,53.110179,40.000000,-13.110179
298,2015-12-27 17:00:00,53.695102,41.666667,-12.028435
